# 🎯 Mecanismos de Atención (Attention Mechanisms)

## De la Atención Básica al Self-Attention y Más Allá

**Objetivo**: Comprender en profundidad todos los tipos de mecanismos de atención, 
su evolución histórica, fundamentos matemáticos e implementación desde cero con PyTorch.

### Contenido
1. **Motivación**: ¿Por qué necesitamos atención?
2. **Atención en Seq2Seq** (Bahdanau, Luong) — el origen
3. **Self-Attention** — cada elemento atiende a todos los demás
4. **Scaled Dot-Product Attention** — la versión del Transformer
5. **Multi-Head Attention** — múltiples perspectivas simultáneas
6. **Variantes**: Causal, Cross, Relative Position, Sparse, Flash
7. **Ejemplo numérico** paso a paso
8. **Implementación from scratch** de cada variante
9. **Experimentos y visualizaciones**

## 1. Motivación: ¿Por qué Atención?

### El problema del cuello de botella

En los modelos Seq2Seq clásicos (Encoder-Decoder con RNN), **toda la información 
de la entrada se comprime en un único vector** de contexto:

```
SIN ATENCIÓN:
    "El gato se sentó en la alfombra"
         ↓   ↓   ↓   ↓   ↓   ↓
    [ h₁ → h₂ → h₃ → h₄ → h₅ → h₆ ] ──→ [c] ──→ Decoder
                                             │
                                    Un solo vector c
                                    contiene TODO
                                             │
                                    Se pierde información
                                    en secuencias largas
```

### La solución: Atención

En lugar de comprimir todo en un vector, el decoder puede **"mirar" directamente** 
a cada estado oculto del encoder:

```
CON ATENCIÓN:
    "El gato se sentó en la alfombra"
         ↓   ↓   ↓   ↓   ↓   ↓
    [ h₁   h₂   h₃   h₄   h₅   h₆ ]
       ↑    ↑    ↑    ↑    ↑    ↑
       │    │    │    │    │    │     ← El decoder "atiende"
       └────┼────┼────┼────┼────┘       selectivamente a
            └────┼────┼────┘             cada estado del encoder
                 └────┘
                   ↓
              "The cat sat on the mat"
```

### Analogía intuitiva

Imagina que estás traduciendo un libro:
- **Sin atención**: Lees todo el libro, cierras el libro, y escribes la traducción de memoria
- **Con atención**: Mientras traduces cada frase, puedes volver a mirar las partes relevantes del original

### Evolución histórica

```
2014 │ Bahdanau Attention ─── Atención aditiva para Seq2Seq
     │   (Additive)            "Neural Machine Translation by Jointly
     │                          Learning to Align and Translate"
     │
2015 │ Luong Attention ─────── Atención multiplicativa, más eficiente
     │   (Multiplicative)       
     │
2017 │ Self-Attention ──────── "Attention Is All You Need" (Transformer)
     │   (Scaled Dot-Product)   Elimina la recurrencia por completo
     │
2018+│ Variantes ──────────── Multi-Head, Sparse, Linear, Flash Attention
```

## 2. Atención en Seq2Seq: Bahdanau y Luong

### 2.1 Atención de Bahdanau (Aditiva, 2014)

El decoder, en cada paso $t$, calcula un score de atención con cada estado del encoder:

**Score** (red neuronal aditiva):
$$e_{t,i} = v^T \tanh(W_1 s_{t-1} + W_2 h_i)$$

**Pesos de atención** (softmax sobre los scores):
$$\alpha_{t,i} = \frac{\exp(e_{t,i})}{\sum_{j=1}^{T_x} \exp(e_{t,j})}$$

**Vector de contexto** (promedio ponderado de los estados del encoder):
$$c_t = \sum_{i=1}^{T_x} \alpha_{t,i} \cdot h_i$$

Donde:
- $s_{t-1}$: estado oculto anterior del decoder
- $h_i$: estado oculto $i$-ésimo del encoder
- $W_1, W_2, v$: parámetros aprendibles

### 2.2 Atención de Luong (Multiplicativa, 2015)

Simplifica el cálculo del score usando un producto punto:

| Variante | Fórmula del Score | Complejidad |
|----------|------------------|-------------|
| **Dot** | $e_{t,i} = s_t^T h_i$ | Más rápida, sin parámetros extra |
| **General** | $e_{t,i} = s_t^T W h_i$ | Un parámetro $W$ para transformar |
| **Concat** | $e_{t,i} = v^T \tanh(W[s_t; h_i])$ | Similar a Bahdanau |

Luego se aplica el mismo softmax y promedio ponderado.

**¿Cuál es mejor?** En la práctica, la atención **dot-product** funciona igual de bien 
y es mucho más eficiente computacionalmente (se puede hacer con multiplicación de matrices).

## 3. Self-Attention: La Revolución

### Del Seq2Seq al Self-Attention

En Bahdanau/Luong, la atención es entre el **decoder y el encoder** (dos secuencias diferentes).

El **Self-Attention** aplica atención **dentro de la misma secuencia**: 
cada token atiende a todos los demás tokens de la misma secuencia.

```
Atención Seq2Seq:                 Self-Attention:
Encoder → Decoder                 Misma secuencia → Misma secuencia
   h₁ ←── s₁                        x₁ ←──→ x₁
   h₂ ←── s₁                        x₁ ←──→ x₂
   h₃ ←── s₁                        x₁ ←──→ x₃
(encoder)  (decoder)                 (misma secuencia)
```

### ¿Por qué Self-Attention?

```
"The animal didn't cross the street because it was too tired"
                                          ↑
                                         "it" = ?
```

Para entender que "it" se refiere a "animal" (y no a "street"), el modelo necesita 
capturar **relaciones de largo alcance** dentro de la misma oración.

| Modelo | Distancia máxima | Paralelización | Complejidad |
|--------|-----------------|---------------|-------------|
| RNN | O(n) pasos | ❌ Secuencial | O(n) |
| CNN | O(n/k) capas | ✅ Paralelo | O(n·k²) |
| **Self-Attention** | **O(1) directo** | **✅ Paralelo** | **O(n²·d)** |

El self-attention conecta **cualquier par de tokens directamente** en un solo paso.

## 4. Scaled Dot-Product Attention

### La fórmula fundamental

Cada token genera tres vectores a partir de su embedding:

- **Query (Q)**: "¿Qué estoy buscando?"
- **Key (K)**: "¿Qué información tengo para ofrecer?"
- **Value (V)**: "¿Cuál es mi contenido?"

$$Q = XW^Q, \quad K = XW^K, \quad V = XW^V$$

La atención se calcula como:

$$\boxed{\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V}$$

### Desglose paso a paso:

**Paso 1** — Similitud: $S = QK^T \in \mathbb{R}^{n \times n}$
$$S_{ij} = q_i \cdot k_j = \sum_{d=1}^{d_k} q_{i,d} \cdot k_{j,d}$$
Mide cuánto el token $i$ (query) es compatible con el token $j$ (key).

**Paso 2** — Escalado: $S' = S / \sqrt{d_k}$

¿Por qué escalar? Si $q, k \sim \mathcal{N}(0, 1)$:
$$\text{Var}(q \cdot k) = d_k$$
$$\therefore \text{Var}(q \cdot k / \sqrt{d_k}) = 1$$

Sin escalar, los valores son grandes → softmax produce distribuciones muy "picudas" 
→ gradientes muy pequeños (saturación).

**Paso 3** — Softmax: $\alpha_{ij} = \text{softmax}_j(S'_i)$
$$\alpha_{ij} = \frac{\exp(S'_{ij})}{\sum_{k=1}^{n} \exp(S'_{ik})}$$
Convierte los scores en **probabilidades** (pesos de atención que suman 1).

**Paso 4** — Agregación: $\text{output}_i = \sum_j \alpha_{ij} \cdot v_j$

El output de cada token es un **promedio ponderado** de los values, 
donde los pesos están determinados por la similitud query-key.

## 5. Multi-Head Attention

### ¿Por qué múltiples cabezas?

Una sola cabeza de atención captura **un tipo de relación**. 
Con múltiples cabezas, el modelo puede capturar **relaciones diversas simultáneamente**:

```
Cabeza 1: Relaciones sintácticas      "gato" ←→ "come" (sujeto-verbo)
Cabeza 2: Relaciones de proximidad     "gato" ←→ "el" (adyacente)
Cabeza 3: Relaciones semánticas        "gato" ←→ "animal" (significado)
Cabeza 4: Relaciones posicionales      "gato" ←→ posición 2
```

### Fórmula

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) W^O$$

$$\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

Donde:
- $W_i^Q, W_i^K \in \mathbb{R}^{d_{model} \times d_k}$, con $d_k = d_{model} / h$
- $W_i^V \in \mathbb{R}^{d_{model} \times d_v}$, con $d_v = d_{model} / h$
- $W^O \in \mathbb{R}^{h \cdot d_v \times d_{model}}$

### Eficiencia

El costo computacional es **igual** al de una sola cabeza con dimensión $d_{model}$, 
porque dividimos la dimensión entre las cabezas:

$$h \times (n^2 \cdot d_k) = h \times \left(n^2 \cdot \frac{d_{model}}{h}\right) = n^2 \cdot d_{model}$$

## 6. Variantes de Atención

### 6.1 Atención Causal (Masked)

Usada en modelos autoregresivos (GPT). Cada token **solo puede ver los tokens anteriores**:

$$\text{CausalAttn}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + M\right) V$$

Donde la máscara $M$:
$$M_{ij} = \begin{cases} 0 & \text{si } j \leq i \quad (\text{puede ver}) \\ -\infty & \text{si } j > i \quad (\text{no puede ver futuro}) \end{cases}$$

### 6.2 Cross-Attention

Usada en el decoder del Transformer original. Las queries vienen del **decoder** 
y las keys/values del **encoder**:

$$\text{CrossAttn}(Q_{dec}, K_{enc}, V_{enc})$$

### 6.3 Atención con Posición Relativa

En lugar de posiciones absolutas, usa la **distancia relativa** entre tokens:

$$e_{ij} = \frac{q_i \cdot k_j + q_i \cdot r_{j-i}}{\sqrt{d_k}}$$

Donde $r_{j-i}$ es un embedding aprendido de la distancia relativa.

### 6.4 Atención Sparse

Para secuencias largas ($n$ grande), la atención $O(n^2)$ es costosa. 
Soluciones:
- **Local Attention**: solo atiende a una ventana de tokens cercanos
- **Strided Attention**: atiende cada $k$ tokens
- **Longformer**: combinación de local + global
- **Linear Attention**: aproxima softmax para lograr $O(n)$

### 6.5 Flash Attention (2022)

No cambia la **matemática** sino la **implementación**:
- Usa tiling para optimizar accesos a memoria GPU
- $2-4\times$ más rápido, usa menos memoria
- Resultado **exactamente igual** al attention estándar

### Resumen de variantes

| Variante | Complejidad | Direccionalidad | Uso |
|----------|------------|----------------|-----|
| Full Self-Attention | O(n²) | Bidireccional | BERT |
| Causal (Masked) | O(n²) | Unidireccional | GPT |
| Cross-Attention | O(n·m) | Encoder→Decoder | Traducción, T5 |
| Relative Position | O(n²) | Bidireccional | T5, DeBERTa |
| Sparse/Local | O(n·k) | Configurable | Longformer, BigBird |
| Linear | O(n) | Configurable | Performer |

## 7. Ejemplo Numérico Paso a Paso

Veamos la atención completa con números reales.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

print("=" * 70)
print("EJEMPLO NUMÉRICO COMPLETO: SELF-ATTENTION")
print("=" * 70)

# --- Configuración ---
# Secuencia: "El gato come pescado" (4 tokens)
# Embeddings de dimensión 6 (simplificado)
tokens = ["El", "gato", "come", "pescado"]
n_tokens = 4
d_model = 6

# Embeddings (valores inventados pero intuitivos)
X = np.array([
    [1.0, 0.2, 0.0, 0.1, 0.5, 0.3],   # "El"      (artículo)
    [0.3, 1.0, 0.8, 0.1, 0.2, 0.9],   # "gato"    (sustantivo, animal)
    [0.1, 0.3, 0.2, 1.0, 0.8, 0.1],   # "come"    (verbo)
    [0.2, 0.9, 0.7, 0.3, 0.1, 0.8],   # "pescado" (sustantivo, comida)
])
print(f"
Embeddings X ({n_tokens} tokens × {d_model} dims):")
for i, token in enumerate(tokens):
    print(f"  {token:>8}: {X[i]}")

# --- Matrices de proyección (normalmente aprendidas) ---
np.random.seed(42)
d_k = 4  # dimensión de Q, K, V
W_Q = np.random.randn(d_model, d_k) * 0.5
W_K = np.random.randn(d_model, d_k) * 0.5
W_V = np.random.randn(d_model, d_k) * 0.5

# --- PASO 1: Calcular Q, K, V ---
Q = X @ W_Q
K = X @ W_K
V = X @ W_V

print(f"
{'='*70}")
print("PASO 1: Proyecciones Q = X·W_Q, K = X·W_K, V = X·W_V")
print(f"{'='*70}")
print(f"  W_Q shape: {W_Q.shape} (d_model × d_k)")
print(f"
  Q (Queries — '¿qué busco?'):")
for i, token in enumerate(tokens):
    print(f"    {token:>8}: {Q[i]}")
print(f"
  K (Keys — '¿qué ofrezco?'):")
for i, token in enumerate(tokens):
    print(f"    {token:>8}: {K[i]}")
print(f"
  V (Values — '¿cuál es mi contenido?'):")
for i, token in enumerate(tokens):
    print(f"    {token:>8}: {V[i]}")

# --- PASO 2: Scores = Q · K^T ---
scores = Q @ K.T
print(f"
{'='*70}")
print("PASO 2: Scores = Q · K^T  (similitud entre cada par)")
print(f"{'='*70}")
print(f"  Scores ({n_tokens}×{n_tokens}):")
print(f"  {'':>10} ", end="")
for t in tokens: print(f"{t:>10}", end="")
print()
for i, token in enumerate(tokens):
    print(f"  {token:>10} ", end="")
    for j in range(n_tokens):
        print(f"{scores[i][j]:10.4f}", end="")
    print()

# --- PASO 3: Escalado ---
scores_scaled = scores / np.sqrt(d_k)
print(f"
{'='*70}")
print(f"PASO 3: Escalado / √d_k = / √{d_k} = / {np.sqrt(d_k):.2f}")
print(f"{'='*70}")
print(f"  Scores escalados:")
for i, token in enumerate(tokens):
    print(f"  {token:>10}: {scores_scaled[i]}")

# --- PASO 4: Softmax ---
def softmax(x):
    e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e_x / e_x.sum(axis=-1, keepdims=True)

attention_weights = softmax(scores_scaled)
print(f"
{'='*70}")
print("PASO 4: Softmax → Pesos de atención (cada fila suma 1)")
print(f"{'='*70}")
for i, token in enumerate(tokens):
    row = attention_weights[i]
    max_j = np.argmax(row)
    print(f"  {token:>10} → ", end="")
    for j in range(n_tokens):
        marker = " ★" if j == max_j else "  "
        print(f"{tokens[j]}:{row[j]:.3f}{marker}", end="  ")
    print()

# --- PASO 5: Output = α · V ---
output = attention_weights @ V
print(f"
{'='*70}")
print("PASO 5: Output = α · V  (promedio ponderado de Values)")
print(f"{'='*70}")
for i, token in enumerate(tokens):
    print(f"  {token:>10}: {output[i]}")
    
print(f"
💡 Interpretación:")
for i, token in enumerate(tokens):
    top_idx = np.argsort(attention_weights[i])[::-1][:2]
    print(f"  '{token}' atiende más a: '{tokens[top_idx[0]]}' ({attention_weights[i][top_idx[0]]:.1%}) "
          f"y '{tokens[top_idx[1]]}' ({attention_weights[i][top_idx[1]]:.1%})")

In [ ]:
# --- Visualización completa del proceso ---
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# 1. Mapa de calor de Scores (sin escalar)
im1 = axes[0, 0].imshow(scores, cmap='RdBu_r', aspect='auto')
axes[0, 0].set_xticks(range(4)); axes[0, 0].set_xticklabels(tokens, fontsize=11)
axes[0, 0].set_yticks(range(4)); axes[0, 0].set_yticklabels(tokens, fontsize=11)
axes[0, 0].set_title('Paso 2: Scores = Q·Kᵀ', fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel('Key'); axes[0, 0].set_ylabel('Query')
for i in range(4):
    for j in range(4):
        axes[0, 0].text(j, i, f'{scores[i][j]:.2f}', ha='center', va='center', fontsize=10)
plt.colorbar(im1, ax=axes[0, 0])

# 2. Scores escalados
im2 = axes[0, 1].imshow(scores_scaled, cmap='RdBu_r', aspect='auto')
axes[0, 1].set_xticks(range(4)); axes[0, 1].set_xticklabels(tokens, fontsize=11)
axes[0, 1].set_yticks(range(4)); axes[0, 1].set_yticklabels(tokens, fontsize=11)
axes[0, 1].set_title(f'Paso 3: Scores / √d_k (÷{np.sqrt(d_k):.2f})', fontsize=13, fontweight='bold')
for i in range(4):
    for j in range(4):
        axes[0, 1].text(j, i, f'{scores_scaled[i][j]:.2f}', ha='center', va='center', fontsize=10)
plt.colorbar(im2, ax=axes[0, 1])

# 3. Attention weights
im3 = axes[0, 2].imshow(attention_weights, cmap='Blues', vmin=0, vmax=attention_weights.max())
axes[0, 2].set_xticks(range(4)); axes[0, 2].set_xticklabels(tokens, fontsize=11)
axes[0, 2].set_yticks(range(4)); axes[0, 2].set_yticklabels(tokens, fontsize=11)
axes[0, 2].set_title('Paso 4: Softmax → α', fontsize=13, fontweight='bold')
axes[0, 2].set_xlabel('Key (a quién atiende)'); axes[0, 2].set_ylabel('Query (quién pregunta)')
for i in range(4):
    for j in range(4):
        color = 'white' if attention_weights[i][j] > 0.35 else 'black'
        axes[0, 2].text(j, i, f'{attention_weights[i][j]:.3f}', 
                        ha='center', va='center', fontsize=10, color=color)
plt.colorbar(im3, ax=axes[0, 2])

# 4. Efecto del escalado en softmax
scales = [0.5, 1.0, 2.0, 4.0]
sample_scores = scores[1]  # scores de "gato"
for s in scales:
    scaled = sample_scores / s
    probs = softmax(scaled)
    axes[1, 0].plot(probs, 'o-', label=f'÷{s}', markersize=8, linewidth=2)
axes[1, 0].set_xticks(range(4)); axes[1, 0].set_xticklabels(tokens)
axes[1, 0].set_title('Efecto del Escalado en Softmax
(Query: "gato")', fontsize=13, fontweight='bold')
axes[1, 0].set_ylabel('Probabilidad')
axes[1, 0].legend(title='Divisor')
axes[1, 0].grid(True, alpha=0.3)

# 5. Comparar Q, K, V para cada token
x_pos = np.arange(d_k)
width = 0.2
for idx, token in enumerate(tokens):
    axes[1, 1].bar(x_pos + idx*width, np.abs(Q[idx]), width, alpha=0.8, label=f'{token}')
axes[1, 1].set_xlabel('Dimensión')
axes[1, 1].set_ylabel('|Q|')
axes[1, 1].set_title('Vectores Query por Token', fontsize=13, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].set_xticks(x_pos + width*1.5)
axes[1, 1].set_xticklabels([f'd{i}' for i in range(d_k)])

# 6. Flujo de atención como grafo
axes[1, 2].set_xlim(-0.5, 3.5)
axes[1, 2].set_ylim(-0.5, 3.5)
for i in range(4):
    for j in range(4):
        if attention_weights[i][j] > 0.15:
            axes[1, 2].annotate('',
                xy=(j, 3-j*0.0), xytext=(i, 3-i*0.0),  
                arrowprops=dict(arrowstyle='->', 
                               lw=attention_weights[i][j]*8,
                               alpha=attention_weights[i][j],
                               color='steelblue'))

for i, token in enumerate(tokens):
    axes[1, 2].scatter(i, 3-i*0.0, s=800, c='white', edgecolors='steelblue', linewidth=2, zorder=5)
    axes[1, 2].text(i, 3-i*0.0, token, ha='center', va='center', fontsize=10, fontweight='bold', zorder=6)
axes[1, 2].set_title('Flujo de Atención
(grosor ∝ peso)', fontsize=13, fontweight='bold')
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

## 8. Ejemplo: Atención Causal vs Completa

La diferencia clave entre BERT (bidireccional) y GPT (causal).

In [ ]:
# --- Comparar atención completa vs causal ---
print("=" * 70)
print("ATENCIÓN COMPLETA vs CAUSAL (Masked)")
print("=" * 70)

# Mismos scores del ejemplo anterior
print("
Scores escalados (iguales en ambos casos):")
print(scores_scaled.round(4))

# Máscara causal
causal_mask = np.tril(np.ones((4, 4)))
print(f"
Máscara causal (triangular inferior):")
print(causal_mask.astype(int))

# Aplicar máscara (-inf donde mask=0)
scores_masked = scores_scaled.copy()
scores_masked[causal_mask == 0] = -1e9
print(f"
Scores con máscara causal (futuro = -∞):")
for i, token in enumerate(tokens):
    row = scores_masked[i]
    print(f"  {token:>8}: [{', '.join(f'{v:7.2f}' if v > -1e8 else '  -∞  ' for v in row)}]")

# Softmax con máscara
attn_causal = softmax(scores_masked)
print(f"
Pesos de atención CAUSAL:")
for i, token in enumerate(tokens):
    row = attn_causal[i]
    print(f"  {token:>8} →", end="")
    for j in range(4):
        if j <= i:
            print(f"  {tokens[j]}:{row[j]:.3f}", end="")
        else:
            print(f"  {tokens[j]}: ----", end="")
    print()

# Visualización lado a lado
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Full attention
im1 = axes[0].imshow(attention_weights, cmap='Blues', vmin=0, vmax=0.5)
axes[0].set_title('Full Attention (BERT)
Bidireccional ← →', fontsize=13, fontweight='bold')
for i in range(4):
    for j in range(4):
        axes[0].text(j, i, f'{attention_weights[i][j]:.3f}', 
                    ha='center', va='center', fontsize=11,
                    color='white' if attention_weights[i][j] > 0.35 else 'black')

# Causal attention
im2 = axes[1].imshow(attn_causal, cmap='Oranges', vmin=0, vmax=0.5)
axes[1].set_title('Causal Attention (GPT)
Unidireccional →', fontsize=13, fontweight='bold')
for i in range(4):
    for j in range(4):
        if j <= i:
            axes[1].text(j, i, f'{attn_causal[i][j]:.3f}', 
                        ha='center', va='center', fontsize=11,
                        color='white' if attn_causal[i][j] > 0.35 else 'black')
        else:
            axes[1].text(j, i, '✗', ha='center', va='center', fontsize=14, color='lightgray')

# Mask visualization
mask_visual = np.where(causal_mask == 1, 1, 0.2)
im3 = axes[2].imshow(mask_visual, cmap='RdYlGn', vmin=0, vmax=1)
axes[2].set_title('Máscara Causal
(verde=puede ver, rojo=bloqueado)', fontsize=13, fontweight='bold')
for i in range(4):
    for j in range(4):
        sym = '✓ Puede' if j <= i else '✗ Futuro'
        color = 'darkgreen' if j <= i else 'darkred'
        axes[2].text(j, i, sym, ha='center', va='center', fontsize=9, 
                    color=color, fontweight='bold')

for ax in axes:
    ax.set_xticks(range(4)); ax.set_xticklabels(tokens, fontsize=11)
    ax.set_yticks(range(4)); ax.set_yticklabels(tokens, fontsize=11)
    ax.set_xlabel('Key'); ax.set_ylabel('Query')

plt.tight_layout()
plt.show()

print("
💡 Diferencia clave:")
print("   BERT: 'come' puede ver 'pescado' (bidireccional, mejor comprensión)")
print("   GPT:  'come' NO puede ver 'pescado' (solo pasado, permite generación)")

## 9. Implementación From Scratch con PyTorch

Implementaremos cada tipo de atención como módulo reutilizable.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo: {device}")

# =============================================================================
# 1. Scaled Dot-Product Attention
# =============================================================================

def scaled_dot_product_attention(Q, K, V, mask=None, dropout=None):
    """
    Atención por producto punto escalado.
    
    Args:
        Q: (batch, [heads], seq_q, d_k)
        K: (batch, [heads], seq_k, d_k)
        V: (batch, [heads], seq_k, d_v)
        mask: tensor booleano, True donde se permite atender
        
    Returns:
        output: (batch, [heads], seq_q, d_v)
        weights: (batch, [heads], seq_q, seq_k)
    """
    d_k = Q.size(-1)
    
    # Scores: (batch, heads, seq_q, seq_k)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    
    # Aplicar máscara
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    # Softmax → pesos
    weights = F.softmax(scores, dim=-1)
    
    # Dropout opcional (regularización)
    if dropout is not None:
        weights = dropout(weights)
    
    # Output: promedio ponderado de V
    output = torch.matmul(weights, V)
    
    return output, weights

print("✅ scaled_dot_product_attention implementada")

# Verificación rápida
Q = torch.randn(2, 4, 3)  # batch=2, seq=4, d_k=3
K = torch.randn(2, 4, 3)
V = torch.randn(2, 4, 3)
out, w = scaled_dot_product_attention(Q, K, V)
print(f"   Input: Q{list(Q.shape)}, K{list(K.shape)}, V{list(V.shape)}")
print(f"   Output: {list(out.shape)}, Weights: {list(w.shape)}")
print(f"   Weights sum: {w.sum(-1)[0]}")  # debe ser ~1.0

In [ ]:
# =============================================================================
# 2. Atención de Bahdanau (Aditiva)
# =============================================================================

class BahdanauAttention(nn.Module):
    """
    Atención aditiva de Bahdanau (2014).
    
    score(s_t, h_i) = v^T · tanh(W1·s_t + W2·h_i)
    
    Usada en Seq2Seq: s_t es del decoder, h_i son del encoder.
    """
    
    def __init__(self, hidden_dim):
        super().__init__()
        self.W1 = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W2 = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.v = nn.Linear(hidden_dim, 1, bias=False)
    
    def forward(self, decoder_state, encoder_outputs):
        """
        Args:
            decoder_state: (batch, hidden_dim) — estado actual del decoder
            encoder_outputs: (batch, src_len, hidden_dim) — todos los estados del encoder
        Returns:
            context: (batch, hidden_dim) — vector de contexto
            weights: (batch, src_len) — pesos de atención
        """
        # decoder_state: (batch, 1, hidden) para broadcasting
        s = decoder_state.unsqueeze(1)
        
        # Score: v^T · tanh(W1·s + W2·h)
        energy = torch.tanh(self.W1(s) + self.W2(encoder_outputs))  # (batch, src_len, hidden)
        scores = self.v(energy).squeeze(-1)  # (batch, src_len)
        
        # Softmax
        weights = F.softmax(scores, dim=-1)  # (batch, src_len)
        
        # Contexto: promedio ponderado
        context = torch.bmm(weights.unsqueeze(1), encoder_outputs).squeeze(1)  # (batch, hidden)
        
        return context, weights

# Verificación
hidden_dim = 16
bahdanau = BahdanauAttention(hidden_dim)
dec_state = torch.randn(2, hidden_dim)          # batch=2
enc_outputs = torch.randn(2, 10, hidden_dim)    # src_len=10
ctx, w = bahdanau(dec_state, enc_outputs)
print(f"\n✅ Bahdanau Attention:")
print(f"   Decoder state: {list(dec_state.shape)}")
print(f"   Encoder outputs: {list(enc_outputs.shape)}")
print(f"   Context: {list(ctx.shape)}, Weights: {list(w.shape)}")
print(f"   Weights sum: {w.sum(-1)}")

In [ ]:
# =============================================================================
# 3. Atención de Luong (Multiplicativa - 3 variantes)
# =============================================================================

class LuongAttention(nn.Module):
    """
    Atención multiplicativa de Luong (2015).
    
    Tres variantes del score:
    - dot:     score = s_t^T · h_i
    - general: score = s_t^T · W · h_i  
    - concat:  score = v^T · tanh(W · [s_t; h_i])
    """
    
    def __init__(self, hidden_dim, method='dot'):
        super().__init__()
        self.method = method
        self.hidden_dim = hidden_dim
        
        if method == 'general':
            self.W = nn.Linear(hidden_dim, hidden_dim, bias=False)
        elif method == 'concat':
            self.W = nn.Linear(hidden_dim * 2, hidden_dim, bias=False)
            self.v = nn.Linear(hidden_dim, 1, bias=False)
    
    def score(self, decoder_state, encoder_outputs):
        """Calcula scores según el método elegido."""
        if self.method == 'dot':
            # s^T · h
            return torch.bmm(encoder_outputs, decoder_state.unsqueeze(2)).squeeze(2)
        
        elif self.method == 'general':
            # s^T · W · h
            energy = self.W(encoder_outputs)  # (batch, src_len, hidden)
            return torch.bmm(energy, decoder_state.unsqueeze(2)).squeeze(2)
        
        elif self.method == 'concat':
            # v^T · tanh(W · [s; h])
            src_len = encoder_outputs.size(1)
            s = decoder_state.unsqueeze(1).expand(-1, src_len, -1)
            concat = torch.cat([s, encoder_outputs], dim=2)
            energy = torch.tanh(self.W(concat))
            return self.v(energy).squeeze(2)
    
    def forward(self, decoder_state, encoder_outputs):
        scores = self.score(decoder_state, encoder_outputs)
        weights = F.softmax(scores, dim=-1)
        context = torch.bmm(weights.unsqueeze(1), encoder_outputs).squeeze(1)
        return context, weights

# Verificación de las 3 variantes
print("✅ Luong Attention (3 variantes):")
for method in ['dot', 'general', 'concat']:
    luong = LuongAttention(hidden_dim, method=method)
    ctx, w = luong(dec_state, enc_outputs)
    n_params = sum(p.numel() for p in luong.parameters())
    print(f"   {method:>8}: context={list(ctx.shape)}, weights={list(w.shape)}, params={n_params}")

In [ ]:
# =============================================================================
# 4. Multi-Head Self-Attention
# =============================================================================

class MultiHeadSelfAttention(nn.Module):
    """
    Multi-Head Self-Attention (Vaswani et al., 2017).
    
    Cada cabeza aprende diferentes relaciones:
    MultiHead(X) = Concat(head_1, ..., head_h) · W_O
    head_i = Attention(X·W_i^Q, X·W_i^K, X·W_i^V)
    """
    
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model debe ser divisible por n_heads"
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        # Proyecciones Q, K, V (una sola matriz grande, más eficiente)
        self.W_qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        """
        Args:
            x: (batch, seq_len, d_model)
            mask: opcional (batch, 1, seq_len, seq_len) o (1, 1, seq_len, seq_len)
        """
        batch_size, seq_len, _ = x.shape
        
        # 1. Proyectar Q, K, V de una vez (eficiente)
        qkv = self.W_qkv(x)  # (batch, seq, 3*d_model)
        qkv = qkv.reshape(batch_size, seq_len, 3, self.n_heads, self.d_k)
        qkv = qkv.permute(2, 0, 3, 1, 4)  # (3, batch, heads, seq, d_k)
        Q, K, V = qkv[0], qkv[1], qkv[2]
        
        # 2. Scaled dot-product attention
        output, weights = scaled_dot_product_attention(Q, K, V, mask, self.dropout)
        
        # 3. Concatenar cabezas
        output = output.transpose(1, 2).contiguous()  # (batch, seq, heads, d_k)
        output = output.reshape(batch_size, seq_len, self.d_model)
        
        # 4. Proyección de salida
        output = self.W_o(output)
        
        return output, weights

# Verificación
d_model, n_heads = 64, 8
mhsa = MultiHeadSelfAttention(d_model, n_heads)
x = torch.randn(2, 10, d_model)

# Sin máscara (bidireccional, tipo BERT)
out_full, w_full = mhsa(x)

# Con máscara causal (tipo GPT)
causal = torch.tril(torch.ones(10, 10)).unsqueeze(0).unsqueeze(0)
out_causal, w_causal = mhsa(x, mask=causal)

total_params = sum(p.numel() for p in mhsa.parameters())
print(f"\n✅ Multi-Head Self-Attention:")
print(f"   d_model={d_model}, n_heads={n_heads}, d_k={d_model//n_heads}")
print(f"   Parámetros: {total_params:,}")
print(f"   Input:  {list(x.shape)}")
print(f"   Output: {list(out_full.shape)}")
print(f"   Weights shape: {list(w_full.shape)} (batch, heads, q, k)")
print(f"\n   Bidireccional (BERT): ✅")
print(f"   Causal (GPT):         ✅")

In [ ]:
# =============================================================================
# 5. Cross-Attention
# =============================================================================

class CrossAttention(nn.Module):
    """
    Cross-Attention: queries del decoder, keys/values del encoder.
    
    Usado en el decoder del Transformer para "mirar" la salida del encoder.
    """
    
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        # Q viene del decoder, K y V del encoder
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, decoder_input, encoder_output, mask=None):
        """
        Args:
            decoder_input: (batch, tgt_len, d_model) — Queries
            encoder_output: (batch, src_len, d_model) — Keys & Values
        """
        batch_size = decoder_input.size(0)
        tgt_len = decoder_input.size(1)
        src_len = encoder_output.size(1)
        
        # Proyecciones
        Q = self.W_q(decoder_input).view(batch_size, tgt_len, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(encoder_output).view(batch_size, src_len, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(encoder_output).view(batch_size, src_len, self.n_heads, self.d_k).transpose(1, 2)
        
        # Atención
        output, weights = scaled_dot_product_attention(Q, K, V, mask, self.dropout)
        
        # Concatenar y proyectar
        output = output.transpose(1, 2).contiguous().reshape(batch_size, tgt_len, self.d_model)
        output = self.W_o(output)
        
        return output, weights

# Verificación
cross_attn = CrossAttention(d_model=64, n_heads=8)
dec_in = torch.randn(2, 8, 64)   # decoder: 8 tokens
enc_out = torch.randn(2, 12, 64)  # encoder: 12 tokens

out, w = cross_attn(dec_in, enc_out)
print(f"\n✅ Cross-Attention:")
print(f"   Decoder input:  {list(dec_in.shape)} (queries)")
print(f"   Encoder output: {list(enc_out.shape)} (keys & values)")
print(f"   Output:  {list(out.shape)}")
print(f"   Weights: {list(w.shape)} → (batch, heads, tgt_len=8, src_len=12)")

In [ ]:
# =============================================================================
# 6. Relative Position Attention
# =============================================================================

class RelativePositionAttention(nn.Module):
    """
    Atención con posición relativa (Shaw et al., 2018).
    
    Añade embeddings de posición relativa a los scores:
    score(i,j) = q_i · k_j + q_i · r_{j-i}
    
    Donde r_{j-i} es un embedding aprendido de la distancia relativa.
    """
    
    def __init__(self, d_model, n_heads, max_relative_pos=16, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.max_relative_pos = max_relative_pos
        
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        
        # Embeddings de posición relativa (2*max_pos + 1 posiciones posibles)
        num_positions = 2 * max_relative_pos + 1
        self.rel_pos_embedding = nn.Embedding(num_positions, self.d_k)
        
        self.dropout = nn.Dropout(dropout)
    
    def _get_relative_positions(self, seq_len):
        """Genera la matriz de posiciones relativas."""
        positions = torch.arange(seq_len).unsqueeze(0) - torch.arange(seq_len).unsqueeze(1)
        # Clip a [-max_pos, max_pos] y shift a [0, 2*max_pos]
        positions = positions.clamp(-self.max_relative_pos, self.max_relative_pos)
        positions = positions + self.max_relative_pos
        return positions
    
    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape
        
        Q = self.W_q(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        
        # Scores estándar: Q·K^T
        scores_content = torch.matmul(Q, K.transpose(-2, -1))
        
        # Scores de posición relativa: Q·R
        rel_pos = self._get_relative_positions(seq_len).to(x.device)  # (seq, seq)
        rel_emb = self.rel_pos_embedding(rel_pos)  # (seq, seq, d_k)
        
        # Q: (batch, heads, seq_q, d_k) → necesitamos (batch*heads, seq_q, d_k)
        Q_flat = Q.reshape(-1, seq_len, self.d_k)
        scores_pos = torch.bmm(Q_flat, rel_emb.unsqueeze(0).expand(Q_flat.size(0), -1, -1, -1)
                               .reshape(Q_flat.size(0), seq_len, seq_len * self.d_k)
                               .reshape(Q_flat.size(0) * seq_len, seq_len, self.d_k)
                               .reshape(Q_flat.size(0), seq_len, self.d_k).transpose(-2, -1))
        scores_pos = scores_pos.view(batch_size, self.n_heads, seq_len, seq_len)
        
        # Combinar
        scores = (scores_content + scores_pos) / math.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        weights = F.softmax(scores, dim=-1)
        weights = self.dropout(weights)
        output = torch.matmul(weights, V)
        
        output = output.transpose(1, 2).contiguous().reshape(batch_size, seq_len, self.d_model)
        output = self.W_o(output)
        
        return output, weights

# Verificación
rel_attn = RelativePositionAttention(d_model=64, n_heads=8, max_relative_pos=16)
x = torch.randn(2, 20, 64)
out, w = rel_attn(x)
print(f"\n✅ Relative Position Attention:")
print(f"   Input: {list(x.shape)}")
print(f"   Output: {list(out.shape)}")
print(f"   Embeddings de posición relativa: {2*16+1} posiciones")
print(f"   Parámetros: {sum(p.numel() for p in rel_attn.parameters()):,}")

## 10. Ejemplo Práctico: Clasificación de Sentimiento con Atención

Entrenaremos un modelo de clasificación de sentimiento para demostrar 
cómo la atención aprende a enfocarse en las **palabras más relevantes**.

In [ ]:
import torch
import torch.optim as optim
import time

# --- Dataset simple de sentimiento ---
oraciones_positivas = [
    "this movie is great and wonderful",
    "i love this amazing film",
    "excellent movie truly fantastic",
    "wonderful acting great story",
    "best movie ever seen love it",
    "amazing film wonderful experience",
    "great film excellent acting superb",
    "love the movie fantastic story",
    "brilliant movie absolutely wonderful",
    "superb film great performance love",
    "this is an excellent wonderful movie",
    "i really love this great film",
    "fantastic acting wonderful story great",
    "amazing wonderful brilliant movie",
    "the best film ever truly excellent",
]

oraciones_negativas = [
    "this movie is terrible and boring",
    "i hate this awful film",
    "worst movie truly horrible",
    "terrible acting bad story",
    "worst movie ever seen hate it",
    "awful film horrible experience",
    "bad film terrible acting poor",
    "hate the movie awful story",
    "dreadful movie absolutely terrible",
    "poor film bad performance hate",
    "this is a terrible horrible movie",
    "i really hate this bad film",
    "awful acting terrible story bad",
    "horrible dreadful worst movie",
    "the worst film ever truly terrible",
]

# Construir vocabulario
todas = oraciones_positivas + oraciones_negativas
palabras = set()
for o in todas:
    for p in o.split():
        palabras.add(p)

vocab = {'<PAD>': 0, '<UNK>': 1}
for p in sorted(palabras):
    vocab[p] = len(vocab)
inv_vocab = {v: k for k, v in vocab.items()}

VOCAB_SIZE = len(vocab)
MAX_LEN = 8  # max tokens por oración

def tokenizar(oracion, vocab, max_len):
    tokens = [vocab.get(p, 1) for p in oracion.split()][:max_len]
    tokens += [0] * (max_len - len(tokens))  # padding
    return tokens

# Preparar datos
X = []
y = []
textos = []
for o in oraciones_positivas:
    X.append(tokenizar(o, vocab, MAX_LEN))
    y.append(1)  # positivo
    textos.append(o)
for o in oraciones_negativas:
    X.append(tokenizar(o, vocab, MAX_LEN))
    y.append(0)  # negativo
    textos.append(o)

X = torch.LongTensor(X)
y = torch.LongTensor(y)

print(f"Dataset: {len(X)} oraciones ({len(oraciones_positivas)} pos, {len(oraciones_negativas)} neg)")
print(f"Vocabulario: {VOCAB_SIZE} palabras")
print(f"Max longitud: {MAX_LEN}")
print(f"\nEjemplo:")
print(f"  Texto: '{textos[0]}'")
print(f"  Tokens: {X[0].tolist()}")
print(f"  Label: {y[0]} (positivo)")

In [ ]:
# --- Modelo de clasificación con Self-Attention ---

class SentimentAttentionModel(nn.Module):
    """
    Clasificador de sentimiento con Self-Attention.
    
    Embedding → Self-Attention → Pooling → Clasificación
    """
    
    def __init__(self, vocab_size, d_model=32, n_heads=4, n_classes=2, max_len=50):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.attention = MultiHeadSelfAttention(d_model, n_heads, dropout=0.1)
        self.layer_norm = nn.LayerNorm(d_model)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 16),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(16, n_classes)
        )
    
    def forward(self, x):
        # Máscara de padding
        pad_mask = (x != 0).unsqueeze(1).unsqueeze(2).float()  # (batch, 1, 1, seq)
        
        # Embedding
        emb = self.embedding(x)  # (batch, seq, d_model)
        
        # Self-Attention
        attn_out, attn_weights = self.attention(emb, mask=pad_mask)
        attn_out = self.layer_norm(emb + attn_out)  # Residual + LayerNorm
        
        # Mean pooling (ignorar padding)
        mask_expanded = (x != 0).unsqueeze(-1).float()
        pooled = (attn_out * mask_expanded).sum(dim=1) / mask_expanded.sum(dim=1).clamp(min=1)
        
        # Clasificación
        logits = self.classifier(pooled)
        
        return logits, attn_weights

# Entrenar
model = SentimentAttentionModel(VOCAB_SIZE, d_model=32, n_heads=4).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.005)
criterion = nn.CrossEntropyLoss()

losses = []
accuracies = []

print("Entrenando clasificador con Self-Attention...")
for epoch in range(80):
    model.train()
    
    # Shuffle
    perm = torch.randperm(len(X))
    X_shuf, y_shuf = X[perm].to(device), y[perm].to(device)
    
    logits, _ = model(X_shuf)
    loss = criterion(logits, y_shuf)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # Accuracy
    with torch.no_grad():
        pred = logits.argmax(dim=-1)
        acc = (pred == y_shuf).float().mean().item()
    
    losses.append(loss.item())
    accuracies.append(acc)
    
    if (epoch + 1) % 20 == 0:
        print(f"  Época {epoch+1:3d} | Loss: {loss.item():.4f} | Accuracy: {acc:.1%}")

print(f"\nAccuracy final: {accuracies[-1]:.1%}")

In [ ]:
# --- Visualizar los pesos de atención para interpretar el modelo ---
model.eval()

# Seleccionar ejemplos interesantes
ejemplos = [
    "this movie is great and wonderful",
    "i hate this awful film",
    "wonderful acting great story",
    "worst movie ever seen hate it",
]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for idx, texto in enumerate(ejemplos):
    row, col = idx // 2, idx % 2
    
    tokens_ids = torch.LongTensor([tokenizar(texto, vocab, MAX_LEN)]).to(device)
    
    with torch.no_grad():
        logits, attn_weights = model(tokens_ids)
    
    pred = logits.argmax(dim=-1).item()
    prob = F.softmax(logits, dim=-1)[0][pred].item()
    label = "POSITIVO 😊" if pred == 1 else "NEGATIVO 😞"
    
    # Promedio de atención sobre todas las cabezas
    attn = attn_weights[0].mean(dim=0).cpu().numpy()  # (seq, seq)
    
    # Tokens como strings
    words = texto.split() + ['<PAD>'] * (MAX_LEN - len(texto.split()))
    
    # Solo mostrar tokens reales (no padding)
    n_real = len(texto.split())
    attn_real = attn[:n_real, :n_real]
    words_real = words[:n_real]
    
    im = axes[row, col].imshow(attn_real, cmap='Blues', vmin=0)
    axes[row, col].set_xticks(range(n_real))
    axes[row, col].set_yticks(range(n_real))
    axes[row, col].set_xticklabels(words_real, fontsize=10, rotation=45, ha='right')
    axes[row, col].set_yticklabels(words_real, fontsize=10)
    axes[row, col].set_title(f'{label} ({prob:.0%})
"{texto}"', fontsize=12, fontweight='bold')
    axes[row, col].set_xlabel('Key (atiende a)')
    axes[row, col].set_ylabel('Query')
    
    for i in range(n_real):
        for j in range(n_real):
            color = 'white' if attn_real[i][j] > 0.3 else 'black'
            axes[row, col].text(j, i, f'{attn_real[i][j]:.2f}', 
                              ha='center', va='center', fontsize=8, color=color)

plt.suptitle('Pesos de Atención: ¿Qué palabras son importantes?', 
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Análisis de atención promedio por palabra
print("\n📊 Atención recibida por cada palabra (¿quién es más 'atendido'?):")
for texto in ejemplos[:2]:
    tokens_ids = torch.LongTensor([tokenizar(texto, vocab, MAX_LEN)]).to(device)
    with torch.no_grad():
        _, attn_weights = model(tokens_ids)
    
    attn = attn_weights[0].mean(dim=0).cpu().numpy()
    words = texto.split()
    n_real = len(words)
    
    # Atención recibida = suma de la columna
    received = attn[:n_real, :n_real].sum(axis=0)
    received = received / received.sum()
    
    print(f"\n  "{texto}":")
    for w, r in sorted(zip(words, received), key=lambda x: -x[1]):
        bar = "█" * int(r * 30)
        print(f"    {w:>12}: {r:.3f} {bar}")

## 11. Visualización: Cada Cabeza Aprende un Patrón Diferente

In [ ]:
# --- Visualizar cada cabeza individualmente ---
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

texto = "wonderful acting great story"
tokens_ids = torch.LongTensor([tokenizar(texto, vocab, MAX_LEN)]).to(device)

with torch.no_grad():
    _, attn_weights = model(tokens_ids)

words = texto.split()
n_real = len(words)

# Atención por cabeza
attn_per_head = attn_weights[0].cpu().numpy()  # (4 heads, seq, seq)

for h in range(4):
    attn_h = attn_per_head[h, :n_real, :n_real]
    
    im = axes[0, h].imshow(attn_h, cmap='Blues', vmin=0)
    axes[0, h].set_xticks(range(n_real))
    axes[0, h].set_yticks(range(n_real))
    axes[0, h].set_xticklabels(words, fontsize=10, rotation=45)
    axes[0, h].set_yticklabels(words, fontsize=10)
    axes[0, h].set_title(f'Cabeza {h+1}', fontsize=13, fontweight='bold')
    
    for i in range(n_real):
        for j in range(n_real):
            color = 'white' if attn_h[i][j] > 0.3 else 'black'
            axes[0, h].text(j, i, f'{attn_h[i][j]:.2f}', 
                          ha='center', va='center', fontsize=9, color=color)

# Atención de otro ejemplo negativo
texto2 = "i hate this awful film"
tokens_ids2 = torch.LongTensor([tokenizar(texto2, vocab, MAX_LEN)]).to(device)
with torch.no_grad():
    _, attn_weights2 = model(tokens_ids2)

words2 = texto2.split()
n_real2 = len(words2)
attn_per_head2 = attn_weights2[0].cpu().numpy()

for h in range(4):
    attn_h = attn_per_head2[h, :n_real2, :n_real2]
    
    im = axes[1, h].imshow(attn_h, cmap='Oranges', vmin=0)
    axes[1, h].set_xticks(range(n_real2))
    axes[1, h].set_yticks(range(n_real2))
    axes[1, h].set_xticklabels(words2, fontsize=10, rotation=45)
    axes[1, h].set_yticklabels(words2, fontsize=10)
    axes[1, h].set_title(f'Cabeza {h+1}', fontsize=13, fontweight='bold')
    
    for i in range(n_real2):
        for j in range(n_real2):
            color = 'white' if attn_h[i][j] > 0.3 else 'black'
            axes[1, h].text(j, i, f'{attn_h[i][j]:.2f}', 
                          ha='center', va='center', fontsize=9, color=color)

axes[0, 0].set_ylabel(f'POSITIVO 😊\n"{texto}"', fontsize=11, color='steelblue')
axes[1, 0].set_ylabel(f'NEGATIVO 😞\n"{texto2}"', fontsize=11, color='darkorange')

plt.suptitle('Patrones de Atención por Cabeza: Positivo vs Negativo', 
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print("💡 Cada cabeza captura un patrón diferente:")
print("   - Algunas cabezas se enfocan en adjetivos (great, awful)")
print("   - Otras capturan relaciones sintácticas (sustantivo-adjetivo)")
print("   - La diversidad de cabezas es lo que hace poderoso al Multi-Head Attention")

## 12. Experimento: Comparar Diferentes Mecanismos de Atención

In [ ]:
# --- Comparar Bahdanau vs Luong en tarea Seq2Seq simple ---
import time

# Construir un mini encoder-decoder con cada tipo de atención

class Seq2SeqWithAttention(nn.Module):
    """Seq2Seq simple con atención configurable para comparación."""
    
    def __init__(self, vocab_size, hidden_dim, attention_type='bahdanau'):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.encoder_rnn = nn.GRU(hidden_dim, hidden_dim, batch_first=True)
        self.decoder_rnn = nn.GRUCell(hidden_dim + hidden_dim, hidden_dim)
        
        if attention_type == 'bahdanau':
            self.attention = BahdanauAttention(hidden_dim)
        else:
            self.attention = LuongAttention(hidden_dim, method=attention_type)
        
        self.fc_out = nn.Linear(hidden_dim * 2, vocab_size)
    
    def forward(self, src, tgt):
        batch_size = src.size(0)
        tgt_len = tgt.size(1)
        
        # Encoder
        enc_emb = self.embedding(src)
        enc_out, hidden = self.encoder_rnn(enc_emb)
        
        # Decoder con atención
        dec_state = hidden.squeeze(0)
        outputs = []
        
        for t in range(tgt_len):
            dec_emb = self.embedding(tgt[:, t])
            
            # Atención
            context, _ = self.attention(dec_state, enc_out)
            
            # Decoder step
            dec_input = torch.cat([dec_emb, context], dim=1)
            dec_state = self.decoder_rnn(dec_input, dec_state)
            
            # Output
            out = self.fc_out(torch.cat([dec_state, context], dim=1))
            outputs.append(out)
        
        return torch.stack(outputs, dim=1)

# --- Datos: invertir secuencias (como en el notebook de Transformers) ---
VOCAB = 12
SEQ_LEN = 6

def gen_reverse_data(n):
    src = torch.randint(3, VOCAB, (n, SEQ_LEN))
    tgt = src.flip(1)
    return src, tgt

train_src, train_tgt = gen_reverse_data(2000)
test_src, test_tgt = gen_reverse_data(300)

# Entrenar y comparar
attention_types = ['bahdanau', 'dot', 'general']
results = {}

for attn_type in attention_types:
    model = Seq2SeqWithAttention(VOCAB, hidden_dim=32, attention_type=attn_type).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.003)
    criterion = nn.CrossEntropyLoss()
    
    losses = []
    start = time.time()
    
    for epoch in range(40):
        model.train()
        perm = torch.randperm(len(train_src))
        epoch_loss = 0
        
        for i in range(0, len(train_src), 64):
            batch_src = train_src[perm[i:i+64]].to(device)
            batch_tgt = train_tgt[perm[i:i+64]].to(device)
            
            output = model(batch_src, batch_tgt)
            loss = criterion(output.reshape(-1, VOCAB), batch_tgt.reshape(-1))
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        
        losses.append(epoch_loss / (len(train_src) // 64))
    
    # Evaluar
    model.eval()
    correct = 0
    with torch.no_grad():
        output = model(test_src.to(device), test_tgt.to(device))
        pred = output.argmax(dim=-1)
        correct = (pred.cpu() == test_tgt).all(dim=1).sum().item()
    
    elapsed = time.time() - start
    accuracy = correct / len(test_src) * 100
    results[attn_type] = {'losses': losses, 'accuracy': accuracy, 'time': elapsed}
    
    print(f"  {attn_type:>10}: Accuracy={accuracy:.1f}%, Time={elapsed:.1f}s")

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_attn = {'bahdanau': 'steelblue', 'dot': 'orange', 'general': 'green'}

for attn_type, res in results.items():
    axes[0].plot(res['losses'], label=f"{attn_type} ({res['accuracy']:.0f}%)", 
                linewidth=2, color=colors_attn[attn_type])
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss')
axes[0].set_title('Convergencia por Tipo de Atención', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy bars
names = list(results.keys())
accs = [results[n]['accuracy'] for n in names]
times = [results[n]['time'] for n in names]
cs = [colors_attn[n] for n in names]

bars = axes[1].bar(names, accs, color=cs, alpha=0.8)
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Accuracy en Test (Invertir Secuencias)', fontsize=13, fontweight='bold')
for bar, acc, t in zip(bars, accs, times):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{acc:.1f}%\n({t:.1f}s)', ha='center', fontsize=10)
axes[1].set_ylim(0, 110)

plt.tight_layout()
plt.show()

## 13. Complejidad Computacional y Escalabilidad

In [ ]:
# --- Medir complejidad empírica ---
import time

seq_lengths = [32, 64, 128, 256, 512, 1024]
d_model = 64
n_heads = 4
batch = 4

times_full = []
times_sparse = []
memory_full = []

print("Midiendo complejidad de Full Self-Attention:")
print(f"  d_model={d_model}, n_heads={n_heads}, batch={batch}")
print(f"  {'Seq Len':>8} | {'Tiempo (ms)':>12} | {'Scores Shape':>20} | {'O(n²·d)':>10}")
print("-" * 65)

mhsa_test = MultiHeadSelfAttention(d_model, n_heads, dropout=0.0).to(device)
mhsa_test.eval()

for n in seq_lengths:
    x = torch.randn(batch, n, d_model).to(device)
    
    # Warmup
    with torch.no_grad():
        _ = mhsa_test(x)
    
    # Medir tiempo
    times = []
    for _ in range(5):
        start = time.perf_counter()
        with torch.no_grad():
            _, w = mhsa_test(x)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        elapsed = (time.perf_counter() - start) * 1000
        times.append(elapsed)
    
    avg_time = np.mean(times)
    times_full.append(avg_time)
    complexity = n * n * d_model
    
    print(f"  {n:>8} | {avg_time:>10.2f}ms | {str(list(w.shape)):>20} | {complexity:>10,}")

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(seq_lengths, times_full, 'bo-', linewidth=2, markersize=8, label='Medido')

# Ajuste cuadrático teórico
from numpy.polynomial import polynomial as P
coeffs = np.polyfit(seq_lengths, times_full, 2)
x_fit = np.linspace(seq_lengths[0], seq_lengths[-1], 100)
y_fit = np.polyval(coeffs, x_fit)
axes[0].plot(x_fit, y_fit, 'r--', alpha=0.7, label='Ajuste O(n²)')

axes[0].set_xlabel('Longitud de Secuencia (n)', fontsize=12)
axes[0].set_ylabel('Tiempo (ms)', fontsize=12)
axes[0].set_title('Complejidad Temporal: Self-Attention', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Teórico: comparación de modelos
n_vals = np.array(seq_lengths)
rnn_complexity = n_vals * d_model  # O(n·d²) pero secuencial
attn_complexity = n_vals**2 * d_model  # O(n²·d) 
local_complexity = n_vals * 64 * d_model  # O(n·k·d) con ventana k=64

axes[1].plot(n_vals, rnn_complexity / 1e6, 's-', label='RNN: O(n·d²)', linewidth=2)
axes[1].plot(n_vals, attn_complexity / 1e6, 'o-', label='Self-Attn: O(n²·d)', linewidth=2)
axes[1].plot(n_vals, local_complexity / 1e6, '^-', label='Local Attn: O(n·k·d)', linewidth=2)
axes[1].set_xlabel('Longitud de Secuencia (n)', fontsize=12)
axes[1].set_ylabel('Operaciones (×10⁶)', fontsize=12)
axes[1].set_title('Complejidad Teórica Comparada', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

print("\n💡 Insights:")
print("   - Self-Attention es O(n²) → costoso para secuencias largas (>2048 tokens)")
print("   - RNN es O(n) pero SECUENCIAL → no se puede paralelizar")
print("   - Local/Sparse Attention es O(n·k) → mejor para secuencias muy largas")
print("   - Flash Attention no cambia la complejidad, pero optimiza el acceso a memoria")

## 14. Resumen y Conclusiones

### Todos los mecanismos de atención:

| Mecanismo | Fórmula | Año | Uso |
|-----------|---------|-----|-----|
| **Bahdanau** | $v^T \tanh(W_1 s + W_2 h)$ | 2014 | Seq2Seq (traducción) |
| **Luong (dot)** | $s^T h$ | 2015 | Seq2Seq (más eficiente) |
| **Luong (general)** | $s^T W h$ | 2015 | Seq2Seq (con parámetros) |
| **Self-Attention** | $\text{softmax}(QK^T/\sqrt{d_k})V$ | 2017 | Transformer |
| **Multi-Head** | $\text{Concat}(\text{head}_1,...,\text{head}_h)W^O$ | 2017 | Transformer |
| **Causal (Masked)** | Self-Attn + máscara triangular | 2017 | GPT (generación) |
| **Cross-Attention** | Q del decoder, K/V del encoder | 2017 | Traducción, T5 |
| **Relative Position** | + embedding de distancia relativa | 2018 | T5, DeBERTa |
| **Sparse/Local** | Atención solo a ventana local | 2020 | Longformer, BigBird |
| **Flash Attention** | Misma matemática, mejor implementación | 2022 | Todos (optimización) |

### Ideas clave:

1. **Atención = promedio ponderado** donde los pesos dependen del contenido
2. **Self-Attention** permite conexiones directas entre **cualquier** par de tokens
3. **Multi-Head** captura **diferentes tipos** de relaciones simultáneamente
4. **Escalado** ($/ \sqrt{d_k}$) evita saturación del softmax
5. **Máscaras** controlan qué tokens pueden ver: bidireccional (BERT) vs causal (GPT)
6. **Cross-Attention** conecta encoder y decoder en modelos seq2seq
7. La **complejidad O(n²)** es el principal limitante → soluciones sparse/linear

### Jerarquía de los mecanismos:

```
Atención Básica (Bahdanau/Luong)
    └── Self-Attention (sin encoder-decoder separado)
         ├── Scaled Dot-Product (con √d_k)
         │    └── Multi-Head (múltiples cabezas)
         │         ├── Full/Bidireccional → BERT
         │         ├── Causal/Masked → GPT
         │         └── Cross-Attention → Traducción
         └── Variantes de eficiencia
              ├── Sparse → Longformer
              ├── Linear → Performer
              └── Flash → Optimización de memoria
```

---
*Notebook creado como parte de la serie "ML from Scratch"*